# Experimentos

## Importando requisitos

In [ ]:
import pandas
import os
import numpy

## Abrindo a planilha

In [ ]:
planilha = pandas.read_csv('../Dataset/DataSummary.csv')
planilha

## Filtrando na planilha os datasets selecionados

In [ ]:
planilha = planilha[planilha['ID'].between(97, 128)]
planilha

## Realizando experimentos no dataset da primeira linha da tabela

Obtendo o caminho do dataset de exemplo

In [ ]:
dataset_exemplo = planilha.iloc[0]['Name']
dataset_exemplo

In [ ]:
dataset_dirname = os.path.join('../Dataset/UCRArchive_2018', dataset_exemplo)
dataset_dirname

In [ ]:
caminho_arquivo_treino = os.path.join(dataset_dirname, '{}_TRAIN.tsv'.format(dataset_exemplo))
caminho_arquivo_treino

In [ ]:
caminho_arquivo_teste = os.path.join(dataset_dirname, '{}_TEST.tsv'.format((dataset_exemplo)))
caminho_arquivo_teste

Abrindo datasets

In [ ]:
dataset_exemplo_treino = pandas.read_csv(caminho_arquivo_treino, sep='\t', header=None)
dataset_exemplo_treino

In [ ]:
dataset_exemplo_teste = pandas.read_csv(caminho_arquivo_teste, sep='\t', header=None)
dataset_exemplo_teste

### Formatando Dataset

In [ ]:
def formatar_dataset(df: pandas.DataFrame) -> pandas.DataFrame:
    classe = df.iloc[:, 0]
    series = df.iloc[:, 1:]

    df_novo = pandas.DataFrame({
        'classe': classe,
        'SérieTemporal': list(series.to_numpy())
    })

    return df_novo

In [ ]:
dataset_exemplo_treino = formatar_dataset(dataset_exemplo_treino)
dataset_exemplo_treino

In [ ]:
dataset_exemplo_teste = formatar_dataset(dataset_exemplo_teste)
dataset_exemplo_teste

### Aplicando algoritmos nos datasets

Importando algoritmos

In [ ]:
from app.model.DynamicTimeWarping import DynamicTimeWarping
from app.model.DerivativeDynamicTimeWarping import DerivativeDynamicTimeWarping
from app.model.LongestCommonSubsequence import LongestCommonSubsequence
from app.model.SoftDynamicTimeWarping import SoftDynamicTimeWarping

Instanciando algoritmos

In [ ]:
dtw = DynamicTimeWarping()
ddtw = DerivativeDynamicTimeWarping()
lcs = LongestCommonSubsequence()
soft_dtw = SoftDynamicTimeWarping()

Definindo série de referência

In [ ]:
serie_referencia = dataset_exemplo_treino.iloc[0]["SérieTemporal"]
serie_referencia

Executando algoritmos e adicionando saída ao Dataset

In [ ]:
dataset_exemplo_treino['dtw'] = dataset_exemplo_treino['SérieTemporal'].apply(
    lambda s: dtw.obter_distancia(s, serie_referencia)
)

In [ ]:
dataset_exemplo_treino['ddtw'] = dataset_exemplo_treino['SérieTemporal'].apply(
    lambda s: ddtw.obter_distancia(s, serie_referencia)
)

In [ ]:
dataset_exemplo_treino['lcs'] = dataset_exemplo_treino['SérieTemporal'].apply(
    lambda s: lcs.lcs(s, serie_referencia)
)

In [ ]:
dataset_exemplo_treino['soft-dtw'] = dataset_exemplo_treino['SérieTemporal'].apply(
    lambda s: soft_dtw.obter_distancia(s, serie_referencia)
)

### Criando conjuntos de teste e treino

Obtendo conjuntos de treino e teste: x e y

In [ ]:
x_train = []
for linha in dataset_exemplo_treino.iloc[:, 1:].values:
    x_train.append(linha)
x_train

In [ ]:
y_train = []
for linha in dataset_exemplo_treino.iloc[:, 0].values:
    y_train.append(linha)
y_train

In [ ]:
x_test = []
for linha in dataset_exemplo_teste.iloc[:, 1:].values:
    x_test.append(linha)
x_test

In [ ]:
y_test = []
for linha in dataset_exemplo_teste.iloc[:, 0].values:
    y_test.append(linha)
y_test

### Treinando modelo

Normalizando os dados

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.fit_transform(x_test)

Criando e treinando modelo

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier()
knn.fit(x_train, y_train)

Realizando predições

In [ ]:
y_pred = knn.predict(x_test)
y_pred

Avaliando modelo

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
acuracia